# scan-to-print: שלב 0 ב-Google Colab

<div dir="rtl">

המחברת הזו בונה מודל תלת־ממדי מהתמונות של כל חפץ, ובודקת אם המידות שלו מספיק מדויקות.

**לפני שמתחילים:**
1. בתפריט **Runtime → Change runtime type** בוחרים **T4 GPU** ושומרים.
2. ב-Google Drive יוצרים את המבנה הזה (מהטלפון: אפליקציית Drive, הכפתור **+** ואז **העלאה**):

```
המחשב שלי (MyDrive)/
  scan-to-print/
    scans.csv
    photos/
      wood_block/   ← כל התמונות של החפץ הראשון
      mug/          ← כל התמונות של החפץ השני
      ...
```

3. הקובץ `scans.csv` (אפשר ליצור אותו ב-Google Sheets ולהוריד כ-CSV). שם החפץ חייב להיות זהה לשם התיקייה:

```
object_id,dim1_mm,dim2_mm,dim3_mm
wood_block,120.4,58.2,31.0
mug,118.0,95.5,95.5
```

4. מריצים את התאים לפי הסדר, עם ▶ או Ctrl+Enter.

**חשוב לדעת:** ב-Colab החינמי לא תמיד יש GPU פנוי, והחיבור מתנתק אחרי זמן בלי פעילות. התוצאות נשמרות ב-Drive אחרי כל חפץ, כך שאם החיבור נפל, מריצים שוב ורק החפצים שלא הסתיימו יעובדו.

</div>

<div dir="rtl">

## 1. בדיקת GPU

</div>

In [ ]:
import subprocess
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True)
if gpu.returncode != 0:
    raise SystemExit("אין GPU. בתפריט Runtime → Change runtime type בחרו T4 GPU והריצו שוב.")
print(gpu.stdout)

<div dir="rtl">

## 2. התקנה (2–3 דקות)

מתקין את הכלי מהמאגר ואת COLMAP עם תמיכה ב-GPU.

</div>

In [ ]:
REF = "main"  # ענף המאגר. אחרי מיזוג ה-PR נשאר main
!pip install -q "scan2print[recon] @ git+https://github.com/d0548450344-dotcom/scan-to-print@{REF}#subdirectory=spike"

import pycolmap
if not pycolmap.has_cuda:
    raise SystemExit("COLMAP הותקן בלי תמיכה ב-GPU. עשו Runtime → Restart session והריצו שוב את התא.")
print("pycolmap", pycolmap.__version__, "עם CUDA")

<div dir="rtl">

## 3. חיבור ל-Google Drive ובדיקת הקבצים

</div>

In [ ]:
import csv
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/scan-to-print")
PHOTOS, RESULTS = ROOT / "photos", ROOT / "results"
RESULTS.mkdir(parents=True, exist_ok=True)

if not (ROOT / "scans.csv").exists():
    raise SystemExit(f"לא נמצא {ROOT / 'scans.csv'}")
with open(ROOT / "scans.csv", newline="", encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))

EXTS = {".jpg", ".jpeg", ".png"}
problems = []
for r in rows:
    folder = PHOTOS / r["object_id"]
    n = len([p for p in folder.glob("*") if p.suffix.lower() in EXTS]) if folder.is_dir() else 0
    status = "✓" if n >= 40 else ("מעט תמונות" if n >= 10 else "חסר")
    print(f"{r['object_id']:<20} {n:>3} תמונות  {status}")
    if n < 10:
        problems.append(r["object_id"])
if problems:
    print("\nחפצים בלי מספיק תמונות ייחשבו ככישלון:", ", ".join(problems))

<div dir="rtl">

## 4. שחזור (כמה דקות לכל חפץ)

כל חפץ נשמר ב-`results/<שם החפץ>/` ב-Drive. חפץ שכבר יש לו תוצאה מדולג. כדי להריץ אותו מחדש, משנים `FORCE = True`.

</div>

In [ ]:
import json, shutil, time, traceback
from scan2print.reconstruct import reconstruct

FORCE = False
MAX_IMAGE_SIZE = 2000  # אם יש שגיאת זיכרון ב-GPU, להוריד ל-1600

for r in rows:
    oid = r["object_id"]
    out = RESULTS / oid
    if (out / "mesh.ply").exists() and not FORCE:
        print(f"{oid}: כבר קיים, מדלג")
        continue
    out.mkdir(exist_ok=True)
    t0 = time.time()
    try:
        rep = reconstruct(PHOTOS / oid, Path("/content/work") / oid, max_image_size=MAX_IMAGE_SIZE)
        shutil.copy(rep.mesh_path, out / "mesh.ply")
        info = {**rep.__dict__, "seconds": round(time.time() - t0)}
        print(f"{oid}: הצליח, {rep.images_registered}/{rep.images_total} תמונות, {info['seconds']} שניות")
        for w in rep.warnings:
            print("   אזהרה:", w)
    except Exception as e:
        info = {"error": str(e), "trace": traceback.format_exc(), "seconds": round(time.time() - t0)}
        print(f"{oid}: נכשל: {e}")
    (out / "report.json").write_text(json.dumps(info, ensure_ascii=False, indent=2))

<div dir="rtl">

## 5. בדיקת דיוק: להמשיך או לעצור

משווה את המידות של המודל למדידות הקליבר. סטייה מותרת: 1 מ"מ לכל 100 מ"מ. כדי להמשיך, לפחות 70% מהחפצים צריכים לעבור.

</div>

In [ ]:
from scan2print import measure

eval_csv = RESULTS / "eval_input.csv"
with open(eval_csv, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["object_id", "mesh_path", "dim1_mm", "dim2_mm", "dim3_mm"])
    w.writeheader()
    for r in rows:
        w.writerow({"object_id": r["object_id"], "mesh_path": f"{r['object_id']}/mesh.ply",
                    **{k: r[k] for k in ("dim1_mm", "dim2_mm", "dim3_mm")}})

s = measure.evaluate_batch(eval_csv, RESULTS / "report.csv")
print(f"עברו {s['passed']} מתוך {s['objects']} חפצים ({s['pass_ratio']:.0%}). נדרש: {s['required_ratio']:.0%}")
print("✅ להמשיך ל-MVP" if s["go"] else "❌ לעצור: הדיוק לא מספיק. צריך לשנות את הגדרת המוצר או את שיטת הצילום")
for o in s["failed_objects"]:
    print(f"  נכשל: {o['object_id']}" + (f" ({o['error']})" if o["error"] else ""))

import pandas as pd
pd.read_csv(RESULTS / "report.csv").round(2)

<div dir="rtl">

## 6. צפייה במודל (לא חובה)

</div>

In [ ]:
import trimesh
OBJECT = rows[0]["object_id"]  # להחליף בשם חפץ אחר
trimesh.load(RESULTS / OBJECT / "mesh.ply").show()

<div dir="rtl">

## 7. קובץ להדפסה (לא חובה)

מזינים מידה אמיתית אחת, והקובץ נשמר ב-Drive.

</div>

In [ ]:
from scan2print import prepare

OBJECT = rows[0]["object_id"]
KNOWN_MM = float(rows[0]["dim1_mm"])  # המידה הארוכה של החפץ, במ"מ
mesh, rep = prepare.repair(prepare.load_mesh(RESULTS / OBJECT / "mesh.ply"))
mesh = prepare.scale_to_known(mesh, KNOWN_MM)
mesh = prepare.flatten_base(mesh, cut_mm=1.0 if mesh.is_watertight else 0.0)
out = prepare.export(mesh, RESULTS / OBJECT / f"{OBJECT}.3mf")
print("נשמר:", out, "| סגור להדפסה:", rep.watertight_after, "| מידות (מ\"מ):", prepare.principal_extents(mesh).round(1))